<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 11. BERT: MLM과 NSP — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

- **원 논문:** [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)
- **저자 / 발표:** Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova · NAACL 2019 (2018)
- **난이도 / 예상 시간:** 중상급 · 약 90분
- **선수 지식:** Transformer encoder, masked loss, segment embedding, multi-task learning

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

token·position·segment embedding을 더한 양방향 Transformer encoder를 만들고, 선택 token의 80/10/10 corruption을 거친 Masked Language Model과 [CLS] 기반 Next Sentence Prediction을 공동 학습한다. 두 metric과 오른쪽 문맥 민감도를 검증한다.

**원 논문과 다른 것**

BERT Base/Large는 각각 12/24층, 110M/340M 파라미터로 BooksCorpus와 English Wikipedia(총 33억 단어)에서 100만 step 사전학습됐다. 여기서는 10개 문장 쌍, 1층 24-hidden encoder를 100여 step 암기시키는 구조 검증이다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3: input representation = token + segment + position embedding | `TinyBERT.embed`의 세 embedding 합과 LayerNorm | hidden 및 두 head logit shape 검사 |
| §3.1, Task #1: Masked LM에서 입력 token의 15%를 선택 | 특수 token을 제외한 60개 중 10개를 고정 선택(미니배치 근사 16.7%) | 선택 수와 특수 token 비선택을 assert |
| §3.1, MLM 선택 token의 80% [MASK], 10% random, 10% unchanged | `corrupted_ids`의 deterministic 8/1/1 corruption | 각 corruption 개수를 정확히 검사 |
| §3.1, Task #2: [CLS]로 IsNext/NotNext 예측 | `nsp_head(hidden[:, 0])`와 binary CE | NSP accuracy 출력 및 assert |
| Figure 1: GPT와 달리 모든 층에서 좌우 문맥을 함께 조건으로 사용 | mask 없는 `BidirectionalSelfAttention`과 오른쪽 token 교란 실험 | right-context sensitivity가 0보다 큰지 assert |
| §3 및 Figure 1: Transformer encoder의 양방향 self-attention | `BidirectionalSelfAttention`과 `BertEncoderBlock`의 post-LN residual | attention `[B,H,L,L]`과 양방향 확률 질량을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\mathcal{L}_{\mathrm{BERT}}=
\mathcal{L}_{\mathrm{MLM}}+\mathcal{L}_{\mathrm{NSP}},\qquad
\mathcal{L}_{\mathrm{MLM}}=-\sum_{i\in\mathcal{M}}
\log p(x_i\mid x_{\setminus\mathcal{M}})
$$

- $\mathcal{M}$은 masking된 token 집합이고 NSP는 두 segment의 연속 여부를 판별합니다.
- bidirectional attention이 mask 좌·우 context를 모두 사용하는지 단방향 기준과 비교합니다.
- token/position/segment embedding, encoder, MLM/NSP head를 별도 모듈로 나눕니다.
- encoder 출력 $[B,T,D]$에서 MLM $[B,T,V]$와 NSP $[B,2]$를 계산합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3: input representation = token + segment + position embedding
- **노트북 구현:** `TinyBERT.embed`의 세 embedding 합과 LayerNorm
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** hidden 및 두 head logit shape 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** BERT Base/Large는 각각 12/24층, 110M/340M 파라미터로 BooksCorpus와 English Wikipedia(총 33억 단어)에서 100만 step 사전학습됐다. 여기서는 10개 문장 쌍, 1층 24-hidden encoder를 100여 step 암기시키는 구조 검증이다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

BERT는 causal mask가 없는 Transformer encoder로 모든 층에서 왼쪽과 오른쪽 문맥을
함께 사용한다. 그대로 다음 token을 예측하면 정답을 보게 되므로, 입력 token 일부를
가리고 원 token을 맞히는 MLM을 쓴다. 선택된 token은 80% `[MASK]`, 10% random,
10% unchanged로 바꾸며, 문장 쌍의 `[CLS]` 표현에는 NSP 이진 목적을 함께 적용한다.

구현에서는 `nn.TransformerEncoder`를 한 줄로 호출하지 않는다. query/key/value
projection, head 분할, scaled score, 전체 위치 softmax를
`BidirectionalSelfAttention`으로 드러낸다. `BertEncoderBlock`은 BERT의 원래
post-LN 순서인 `LayerNorm(x + sublayer(x))`를 attention과 GELU FFN 뒤에 각각
적용한다. causal mask가 없다는 사실이 코드에서 바로 보여야 GPT와의 차이를
설명할 수 있다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 1111
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())
PAD, CLS, SEP, MASK = 0, 1, 2, 3
VOCAB_SIZE = 24

sentence_pairs = [
    ([4, 5, 6], [7, 8, 9], 1),
    ([7, 8, 9], [13, 14, 15], 1),
    ([10, 11, 12], [16, 17, 18], 1),
    ([16, 17, 18], [19, 20, 21], 1),
    ([4, 6, 5], [8, 9, 7], 1),
    ([4, 5, 6], [16, 17, 18], 0),
    ([7, 8, 9], [19, 20, 21], 0),
    ([10, 11, 12], [7, 8, 9], 0),
    ([16, 17, 18], [13, 14, 15], 0),
    ([4, 6, 5], [19, 21, 20], 0),
]
input_ids = torch.tensor([[CLS, *a, SEP, *b, SEP] for a, b, _ in sentence_pairs])
segment_ids = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1]] * len(sentence_pairs))
nsp_labels = torch.tensor([label for _, _, label in sentence_pairs])
input_ids, segment_ids, nsp_labels = ACCELERATOR.move(
    input_ids, segment_ids, nsp_labels
)

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §3.1, Task #1: Masked LM에서 입력 token의 15%를 선택 / §3.1, MLM 선택 token의 80% [MASK], 10% random, 10% unchanged
- **이 셀의 책임:** 특수 token을 제외한 60개 중 10개를 고정 선택(미니배치 근사 16.7%) / `corrupted_ids`의 deterministic 8/1/1 corruption
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** 선택 수와 특수 token 비선택을 assert / 각 corruption 개수를 정확히 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 11-1: 한 행당 content token 하나를 골라 MLM label(-100=무시)을 만드세요.
selected_positions = torch.tensor([1, 2, 3, 5, 6, 7, 1, 2, 5, 6], device=DEVICE)
corrupted_ids = input_ids.clone()
mlm_labels = torch.full_like(input_ids, -100)
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."
# TODO 11-2: 선택 10개를 8개 MASK / 1개 random / 1개 unchanged로 손상하세요.

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3: input representation = token + segment + position embedding / Figure 1: GPT와 달리 모든 층에서 좌우 문맥을 함께 조건으로 사용 / §3 및 Figure 1: Transformer encoder의 양방향 self-attention
- **이 셀의 책임:** `TinyBERT.embed`의 세 embedding 합과 LayerNorm / mask 없는 `BidirectionalSelfAttention`과 오른쪽 token 교란 실험 / `BidirectionalSelfAttention`과 `BertEncoderBlock`의 post-LN residual
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** hidden 및 두 head logit shape 검사 / right-context sensitivity가 0보다 큰지 assert / attention `[B,H,L,L]`과 양방향 확률 질량을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 11-3: mask 없는 multi-head self-attention을 직접 구현하세요.
# TODO 11-4: 세 embedding, encoder block, MLM/NSP head를 조립하세요.
class BidirectionalSelfAttention(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, num_heads):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def _split_heads(self, tensor):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def _merge_heads(self, tensor):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, hidden):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class BertEncoderBlock(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, num_heads, d_ff):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TinyBERT(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, vocab_size, max_length, d_model=24, num_heads=4):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def embed(self, token_ids, token_type_ids):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def masked_language_model_logits(self, hidden):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, token_ids, token_type_ids):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, Task #1: Masked LM에서 입력 token의 15%를 선택 / §3.1, MLM 선택 token의 80% [MASK], 10% random, 10% unchanged / §3.1, Task #2: [CLS]로 IsNext/NotNext 예측
- **이 셀의 책임:** 특수 token을 제외한 60개 중 10개를 고정 선택(미니배치 근사 16.7%) / `corrupted_ids`의 deterministic 8/1/1 corruption / `nsp_head(hidden[:, 0])`와 binary CE
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 선택 수와 특수 token 비선택을 assert / 각 corruption 개수를 정확히 검사 / NSP accuracy 출력 및 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 11-5: 선택 위치 MLM CE와 [CLS] NSP CE를 더해 공동 학습하세요.
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
total_losses, mlm_losses, nsp_losses = [], [], []
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Figure 1: GPT와 달리 모든 층에서 좌우 문맥을 함께 조건으로 사용
- **이 셀의 책임:** mask 없는 `BidirectionalSelfAttention`과 오른쪽 token 교란 실험
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** right-context sensitivity가 0보다 큰지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 11-6: 오른쪽 token을 바꿔 양방향 문맥 민감도를 측정하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 해석 체크

1. MLM loss를 모든 위치가 아니라 선택된 위치에만 적용하는 이유는 무엇인가요?
2. 선택 token의 10%를 그대로 두는 경우에도 정답 loss를 계산하는 이유는 무엇인가요?
3. 이 노트북의 training accuracy가 실제 downstream 일반화를 뜻하지 않는 이유를 적으세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.